In [30]:
# --- STEP 0: IMPORT LIBRARIES ---
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix


In [31]:
# --- STEP 1: LOAD CSV AND PREPARE DATA ---
IMAGE_FOLDER = "Images"   # Folder containing your .tif files
LABELS_FILE = "otolith_labels.csv"

IMG_HEIGHT, IMG_WIDTH = 256, 256
BATCH_SIZE = 8
EPOCHS = 20
LEARNING_RATE = 0.0005

# Load CSV
df = pd.read_csv(LABELS_FILE)
df.columns = ['Image', 'Age', 'Length']  # ensure correct column names

# Fix paths for images
df['Image'] = df['Image'].apply(lambda x: os.path.join(IMAGE_FOLDER, x))

# Treat Age as categorical
df['Age'] = df['Age'].astype(str)

# Encode Age → integer labels
unique_labels = sorted(df['Age'].unique())
label_to_index = {lbl: idx for idx, lbl in enumerate(unique_labels)}
df['Age_encoded'] = df['Age'].map(label_to_index)

print("Unique classes:", unique_labels)
print(df.head())


Unique classes: ['13', '131', '137', '139', '26', '3', '32', '33', '35', '36', '4', '42', '7', '9']
                                          Image Age  Length  Age_encoded
0  Images\Mubar_104mm_13_13-07-16_TAK_MIM_L.tif  13     104            0
1  Images\Mubar_107mm_26_09-07-16_TAK_MIM_L.tif  26     107            4
2   Images\Mubar_195mm_09_03-10-16_MARKET_L.tif   9     195           13
3   Images\Mubar_58mm_09_25-06-16_TAK_MIM_R.tif   9      58           13
4   Images\Mubar_77mm_03_08-10-16_TAK_MIM_L.tif   3      77            5


In [32]:
# --- STEP 2: CREATE TF.DATASET PIPELINE ---

def load_image(file_path, label):
    img = tf.io.read_file(file_path)
    img = tf.image.decode_image(img, channels=3)
    img = tf.image.convert_image_dtype(img, tf.float32)  # normalize to [0,1]
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])
    return img, label

file_paths = df['Image'].values
labels = df['Age'].values

dataset = tf.data.Dataset.from_tensor_slices((file_paths, labels))
dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
dataset = dataset.shuffle(buffer_size=len(df), reshuffle_each_iteration=False)

# Train/Val/Test split
train_size = int(0.7 * len(df))
val_size = int(0.15 * len(df))
test_size = len(df) - train_size - val_size

train_ds = dataset.take(train_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = dataset.skip(train_size).take(val_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = dataset.skip(train_size + val_size).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train size:", train_size, "Val size:", val_size, "Test size:", test_size)


ValueError: in user code:

    File "C:\Users\Apurva gosavi\AppData\Local\Temp\ipykernel_19268\371208043.py", line 7, in load_image  *
        img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH])

    ValueError: 'images' contains no shape.
